# Reusable house-price features with Feast
Train two models from one prepared feature library, then serve both through FastAPI.

**Key distinction:** Python prepares the features once; Feast manages the definitions, historical retrieval, and online access. Run cells in order using the existing `.venv` kernel. See `readme.md` for commands and production caveats.

In [1]:
from pathlib import Path
import sys
import subprocess
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name != 'feature_store_feast':
    ROOT = ROOT / 'feature_store_feast'
assert (ROOT / 'generate_data.py').exists(), 'Start in the project root or feature_store_feast directory'
sys.path.insert(0, str(ROOT))

from generate_data import generate
from build_features import build
from common import store, FEATURE_COLUMNS, SERVICES

print('Python kernel:', sys.executable)

Python kernel: C:\Users\Ash\Desktop\educational\play_ML\.venv\Scripts\python.exe


## 1. Raw data and prediction time
We know sales through timestamp t and predict a sale on t + 1 day. Prices from that future sale are labels, never input features. Repeated sales are a simplified synthetic teaching scenario.

In [2]:
generate()

houses = pd.read_parquet(ROOT / 'data/houses.parquet')
sales  = pd.read_parquet(ROOT / 'data/sales.parquet')
labels = pd.read_parquet(ROOT / 'data/labels.parquet')

display(houses.head(), sales.head(), labels.head())

Generated 300 houses, 6300 sales, 5180 labels in C:\Users\Ash\Desktop\educational\play_ML\feature_store_feast\data


,house_id,neighborhood_id,area_sqft,bedrooms,year_built
0,1,1,1686,4,1970
1,2,5,3004,4,1972
2,3,4,678,4,1980
3,4,3,3204,1,2014
4,5,3,2197,5,1992


,house_id,neighborhood_id,event_timestamp,sale_price
0,204,5,2026-04-27 00:00:00+00:00,776633.099731
1,267,3,2026-04-27 00:00:00+00:00,261063.687983
2,153,6,2026-04-27 00:00:00+00:00,532546.178081
3,10,1,2026-04-27 00:00:00+00:00,116594.404616
4,234,5,2026-04-27 00:00:00+00:00,322520.742115


,house_id,neighborhood_id,sale_timestamp,sale_price,event_timestamp
0,291,5,2026-05-29 00:00:00+00:00,737534.637900,2026-05-28 00:00:00+00:00
1,186,3,2026-05-29 00:00:00+00:00,245726.697288,2026-05-28 00:00:00+00:00
2,229,4,2026-05-29 00:00:00+00:00,744800.357217,2026-05-28 00:00:00+00:00
3,167,2,2026-05-29 00:00:00+00:00,206233.605577,2026-05-28 00:00:00+00:00
4,49,5,2026-05-29 00:00:00+00:00,311358.575873,2026-05-28 00:00:00+00:00


## 2. Engineer once, reuse across consumers
Inspect `build_features.py`: log area, area per bedroom, age, and trailing 30-day neighborhood prices/counts. 

At t the rolling window is (t - 30 days, t]. 

Sales are assumed immediately available. 

This is a batch feature pipeline, not a transformation performed automatically by Feast.

In [3]:
build()

display(pd.read_parquet(ROOT / 'data/house_features.parquet').head())
display(pd.read_parquet(ROOT / 'data/market_features.parquet').tail())

Prepared shared house features and trailing 30-day neighborhood features.


,house_id,area_sqft,bedrooms,log_area,area_per_bedroom,house_age_years,event_timestamp
0,1,1686,4,7.430707,421.5,56,2026-04-27 00:00:00+00:00
1,2,3004,4,8.008033,751.0,54,2026-04-27 00:00:00+00:00
2,3,678,4,6.520621,169.5,46,2026-04-27 00:00:00+00:00
3,4,3204,1,8.072467,3204.0,12,2026-04-27 00:00:00+00:00
4,5,2197,5,7.695303,439.4,34,2026-04-27 00:00:00+00:00


,neighborhood_id,neighborhood_price_per_sqft_30d,neighborhood_sales_count_30d,event_timestamp
1075,2,203.953730,147,2026-10-23 00:00:00+00:00
1076,3,226.336734,234,2026-10-23 00:00:00+00:00
1077,4,249.388332,156,2026-10-23 00:00:00+00:00
1078,5,268.524386,211,2026-10-23 00:00:00+00:00
1079,6,288.918576,129,2026-10-23 00:00:00+00:00


## 3. Register the reusable contracts
Read `feature_repo/definitions.py`. A house entity and a neighborhood entity identify rows. Feature views describe prepared data; two named model feature services reuse those same views. `feast apply` registers metadata and prepares online tables.

In [4]:
subprocess.run([sys.executable, '-c', 'from feast.cli.cli import cli; cli()', 'apply'], cwd=ROOT / 'feature_repo', check=True)

fs = store()

print('Views:', [v.name for v in fs.list_feature_views()])
print('Services:', [s.name for s in fs.list_feature_services()])

Views: ['house_features', 'market_features']
Services: ['forest_price_v1', 'linear_price_v1']


## 4. Retrieve features as they were known historically
Feast joins both views using their entity keys and the prediction timestamp. Each model requests its own service without copying engineering code. The same neighborhood has different market values at different dates.

In [5]:
training_sample = fs.get_historical_features(
    entity_df=labels.head(6),
    features=fs.get_feature_service('linear_price_v1'),
).to_df()

display(training_sample)
market = pd.read_parquet(ROOT / 'data/market_features.parquet')
dates = sorted(market.event_timestamp.unique())
queries = pd.DataFrame({'neighborhood_id': [1, 1], 'event_timestamp': [dates[60], dates[-1]]})

display(fs.get_historical_features(
    entity_df=queries,
    features=['market_features:neighborhood_price_per_sqft_30d']
).to_df())

,house_id,neighborhood_id,sale_timestamp,sale_price,event_timestamp,area_sqft,bedrooms,log_area,area_per_bedroom,house_age_years,neighborhood_price_per_sqft_30d,neighborhood_sales_count_30d
0,167,2,2026-05-29 00:00:00+00:00,206233.605577,2026-05-28 00:00:00+00:00,1123,4,7.024649,280.750000,10,178.636020,137
1,186,3,2026-05-29 00:00:00+00:00,245726.697288,2026-05-28 00:00:00+00:00,1313,3,7.180831,437.666667,15,201.294448,221
2,229,4,2026-05-29 00:00:00+00:00,744800.357217,2026-05-28 00:00:00+00:00,3258,5,8.089176,651.600000,29,227.135347,168
3,49,5,2026-05-29 00:00:00+00:00,311358.575873,2026-05-28 00:00:00+00:00,1165,5,7.061334,233.000000,9,244.171661,227
4,291,5,2026-05-29 00:00:00+00:00,737534.637900,2026-05-28 00:00:00+00:00,2931,1,7.983440,2931.000000,36,244.171661,227
5,269,4,2026-05-29 00:00:00+00:00,567201.270520,2026-05-28 00:00:00+00:00,2481,3,7.816820,827.000000,7,227.135347,168


,neighborhood_id,event_timestamp,neighborhood_price_per_sqft_30d
0,1,2026-06-26 00:00:00+00:00,156.611093
1,1,2026-10-23 00:00:00+00:00,179.907585


## 5. Train two models from the same feature store
Ridge and Random Forest independently retrieve the shared features. The training function asserts identical feature values and uses earlier dates for training and later dates for testing. Feast stores features; scikit-learn trains the estimators.

In [6]:
from train_models import train
train()
import json
display(pd.DataFrame(json.loads((ROOT / 'artifacts/metrics.json').read_text())).T)

{
  "linear": {
    "mae": 30090.997603795655,
    "rmse": 38433.49728392339,
    "train_rows": 4130,
    "test_rows": 1050
  },
  "forest": {
    "mae": 18080.9452480728,
    "rmse": 22651.057423154176,
    "train_rows": 4130,
    "test_rows": 1050
  }
}
Verified: both models trained from identical Feast feature values; test dates follow training dates.


,mae,rmse,train_rows,test_rows
linear,30090.997604,38433.497284,4130.0,1050.0
forest,18080.945248,22651.057423,4130.0,1050.0


## 6. Serve the latest prepared features
Materialization copies the latest snapshots into SQLite. Online lookup uses entity keys and returns latest values; it is not a historical time-travel request. The API sends the returned columns directly to either saved model.

In [7]:
from materialize import materialize
materialize()
keys = {'house_id': 1, 'neighborhood_id': int(houses.set_index('house_id').loc[1, 'neighborhood_id'])}
for model, service in SERVICES.items():
    print(model)
    display(fs.get_online_features(features=fs.get_feature_service(service), entity_rows=[keys]).to_df())

Materializing 2 feature views from 2026-04-27 00:00:00+00:00 to 2026-10-23 00:00:01+00:00 into the sqlite online store.

house_features:
market_features:
linear


,house_id,neighborhood_id,area_per_bedroom,bedrooms,area_sqft,log_area,house_age_years,neighborhood_sales_count_30d,neighborhood_price_per_sqft_30d
0,1,1,421.5,4,1686,7.430707,56,173,179.907585


forest


,house_id,neighborhood_id,area_per_bedroom,bedrooms,area_sqft,log_area,house_age_years,neighborhood_sales_count_30d,neighborhood_price_per_sqft_30d
0,1,1,421.5,4,1686,7.430707,56,173,179.907585


In [8]:
from app import app
from fastapi.testclient import TestClient
with TestClient(app) as client:
    for model in SERVICES:
        result = client.post('/predict', json={'house_id': 1, 'model': model})
        result.raise_for_status()
        display(result.json())

C:\Users\Ash\Desktop\educational\play_ML\.venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


{'house_id': 1,
 'model': 'linear',
 'predicted_price': 294907.76,
 'features': {'area_sqft': 1686.0,
  'bedrooms': 4.0,
  'log_area': 7.430707082545968,
  'area_per_bedroom': 421.5,
  'house_age_years': 56.0,
  'neighborhood_price_per_sqft_30d': 179.9075854491029,
  'neighborhood_sales_count_30d': 173.0}}

{'house_id': 1,
 'model': 'forest',
 'predicted_price': 303894.61,
 'features': {'area_sqft': 1686.0,
  'bedrooms': 4.0,
  'log_area': 7.430707082545968,
  'area_per_bedroom': 421.5,
  'house_age_years': 56.0,
  'neighborhood_price_per_sqft_30d': 179.9075854491029,
  'neighborhood_sales_count_30d': 173.0}}

## 7. Verify consistency and time correctness
Check an old snapshot against raw rolling sales, expired historical lookup behavior, online/offline equality at the latest snapshot, both model API responses, and invalid requests. The local backend can omit expired rows, so training also checks label row counts.

For an actual HTTP server, run `python -m uvicorn app:app` from this directory and visit http://127.0.0.1:8000/docs. FastAPI needs no copy of the engineering logic. Production requires scheduled feature refresh, shared storage, and freshness monitoring; see the README.

In [9]:
from verify_demo import verify
verify()

linear 294907.76


forest 303894.61
PASS: historical snapshot, rolling window, TTL, online/offline parity, and both API models.
